# Exploration des modèles avec LazyPredict

Ce notebook regroupe l'exploration LazyPredict réalisée dans le cadre du projet CO₂. Les résultats du 14 octobre 2026 sont conservés dans un fichier Excel afin de pouvoir les consulter sans réentraîner les modèles.


In [ ]:
import pandas as pd

X_train = pd.read_csv('../data/X_train_final.csv')
X_test = pd.read_csv('../data/X_test_final.csv')
y_train = pd.read_csv('../data/y_train.csv')['co2']
y_test = pd.read_csv('../data/y_test.csv')['co2']


## Exploration automatique des modèles de régression avec LazyRegressor

Afin d'élargir la comparaison à un plus grand nombre d'algorithmes, une exploration automatique est réalisée à l'aide de LazyRegressor. Cet outil entraîne plusieurs modèles de régression avec leurs paramètres par défaut afin d'identifier les approches les plus prometteuses avant une éventuelle phase d'optimisation.

### Préparation de l'environnement d'exécution

Certaines variables d'environnement sont définies afin de limiter le nombre de threads utilisés par les bibliothèques numériques. Cette configuration améliore la stabilité de l'exécution de LazyRegressor dans l'environnement de travail utilisé.

### Exécution facultative

Le code original de LazyPredict est conservé ci-dessous. **Il est désactivé par défaut** car son exécution peut être longue et nécessite les dépendances correspondantes. Les résultats historiques sont consultables dans la cellule suivante.


In [ ]:
EXECUTER_LAZYPREDICT = False

if EXECUTER_LAZYPREDICT:
    import os
    import pandas as pd
    from pathlib import Path
    from lazypredict import LazyRegressor
    from sklearn.preprocessing import MinMaxScaler
    from datetime import datetime

    os.environ["JOBLIB_START_METHOD"] = "fork"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    reg = LazyRegressor(verbose=0, ignore_warnings=True)
    models, predictions = reg.fit(X_train_scaled, X_test_scaled, y_train, y_test)

    models = models.round(4)
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    display(models)

    date = datetime.now().strftime("%Y-%m-%d")
    models.to_excel(Path("..") / "data" / f"LazyPredict_resultats_{date}.xlsx")

### Résultats enregistrés

Lecture du fichier Excel produit lors de l'expérimentation du 14 octobre 2026, sans nouvel entraînement.


In [ ]:
resultats = pd.read_excel('../data/LazyPredict_resultats_2026-10-14.xlsx')
display(resultats)


### Interprétation des résultats de LazyRegressor

L'exploration automatique réalisée à l'aide de LazyRegressor a permis de comparer rapidement un large ensemble de modèles de régression en utilisant leurs paramètres par défaut. Cette approche constitue une première étape exploratoire visant à identifier les algorithmes les plus prometteurs avant une éventuelle phase d'optimisation des hyperparamètres.

Les résultats mettent en évidence les excellentes performances de plusieurs modèles d'ensemble, notamment ExtraTreesRegressor, RandomForestRegressor, BaggingRegressor et ExtraTreeRegressor, qui obtiennent des coefficients de détermination (R²) très élevés ainsi que des erreurs de prédiction (RMSE) particulièrement faibles. Ces résultats montrent que les caractéristiques techniques des véhicules permettent de prédire avec une grande précision les émissions de CO₂.

Les modèles linéaires, tels que LinearRegression, Ridge, RidgeCV et BayesianRidge, obtiennent également d'excellentes performances, avec des coefficients de détermination supérieurs à 0,99. Cela confirme que la relation entre les variables explicatives retenues et les émissions de CO₂ est largement linéaire lorsque les variables les plus informatives sont conservées.

À l'inverse, certains modèles comme DummyRegressor, Lars ou SGDRegressor affichent des performances nettement plus faibles. Ces résultats montrent que ces approches sont moins adaptées à la structure de ce jeu de données ou nécessiteraient un réglage plus fin de leurs hyperparamètres pour atteindre un niveau de performance comparable.

Ces résultats doivent toutefois être interprétés avec prudence. Plusieurs variables de consommation de carburant présentent une corrélation très forte avec la variable cible (CO₂), ce qui peut contribuer à ces excellentes performances. Afin d'évaluer la robustesse des modèles et de limiter le risque de dépendance à des variables fortement corrélées, des expérimentations complémentaires ont été réalisées en supprimant successivement certaines variables telles que conso_mixte, co_typ_1 et nox.

Il convient également de rappeler que LazyRegressor constitue un outil d'aide à la décision et non un outil de sélection définitive du modèle. Son objectif est de fournir une comparaison rapide de nombreux algorithmes afin d'orienter les expérimentations et les analyses plus approfondies.

Enfin, les performances présentées correspondent aux paramètres par défaut des différents algorithmes. Elles devront être confirmées et éventuellement améliorées lors d'une phase d'optimisation des hyperparamètres (GridSearchCV ou RandomizedSearchCV). Le choix du modèle final reposera non seulement sur ses performances prédictives, mais également sur sa robustesse, son interprétabilité, son coût de calcul et sa capacité de généralisation sur des données inédites.